# 故障恢复：重放、查询与并发去重

约 25–30 分钟。纯 Python 标准库、CPU，无需模型密钥；按顺序运行。

第 1、2 节用内存对象模拟远端业务服务与检查点，第 3 节使用本地 SQLite 事务与 20 个并发调用。它们不连接真实订单或支付服务；SQLite 的写事务串行化也不代表多副本数据库的行为。

先预测：远端已提交、本地尚未保存完成状态时，盲目重放会发生什么？比较业务提交次数与网络写请求次数。

对应专题：[Harness](https://suguilin.github.io/study/?id=lesson-harness) / [工具可靠性](https://suguilin.github.io/study/?id=lesson-tool-reliability)。


In [ ]:
from dataclasses import dataclass, field
import json
from pathlib import Path

class SimulatedCrash(Exception):
    pass

@dataclass
class RemoteService:
    idempotent: bool
    query_available: bool = True
    write_requests: int = 0
    commits: list = field(default_factory=list)
    ledger: dict = field(default_factory=dict)

    def write(self, operation_id, payload):
        self.write_requests += 1
        if self.idempotent and operation_id in self.ledger:
            previous = self.ledger[operation_id]
            if previous['payload'] != payload:
                raise ValueError('The same operation ID cannot change its payload')
            return previous
        result = {'operation_id': operation_id, 'payload': payload, 'receipt': len(self.commits) + 1}
        self.commits.append(result)
        self.ledger[operation_id] = result
        return result

    def query(self, operation_id):
        if not self.query_available:
            raise ConnectionError('Query is unavailable; absence has not been established')
        return self.ledger.get(operation_id)

def first_attempt(checkpoint, remote, crash_at):
    checkpoint.update(status='pending', operation_id='task-42-write-1', payload={'artifact': 'note-42'})
    if crash_at == 'before_submit':
        raise SimulatedCrash()
    receipt = remote.write(checkpoint['operation_id'], checkpoint['payload'])
    if crash_at == 'after_remote_commit':
        raise SimulatedCrash()
    checkpoint.update(status='confirmed', receipt=receipt)
    if crash_at == 'after_checkpoint':
        raise SimulatedCrash()

def recover(checkpoint, remote, protocol, reuse_id=True):
    if checkpoint['status'] == 'confirmed':
        return 'confirmed'
    operation_id = checkpoint['operation_id'] if reuse_id else 'new-id-on-retry'
    receipt = None
    if protocol == 'query_then_retry':
        try:
            receipt = remote.query(operation_id)
        except ConnectionError:
            checkpoint['status'] = 'unknown'
            return 'unknown'
    if receipt is None:
        receipt = remote.write(operation_id, checkpoint['payload'])
    checkpoint.update(status='confirmed', receipt=receipt)
    return 'confirmed'

def trial(protocol, crash_at, reuse_id=True, query_available=True):
    remote = RemoteService(idempotent=protocol != 'naive_replay', query_available=query_available)
    checkpoint = {}
    try:
        first_attempt(checkpoint, remote, crash_at)
    except SimulatedCrash:
        pass
    status = recover(checkpoint, remote, protocol, reuse_id)
    return {'protocol': protocol, 'crash_at': crash_at, 'status': status,
            'business_commits': len(remote.commits), 'write_requests': remote.write_requests}


## 1 · 改变崩溃位置

先看 `after_remote_commit`：本地检查点仍是 pending，远端已经提交。运行后比较 **业务提交次数**，再比较请求次数。两个指标不能混淆。

In [ ]:
protocols = ['naive_replay', 'idempotent_replay', 'query_then_retry']
faults = ['before_submit', 'after_remote_commit', 'after_checkpoint']
report = [trial(protocol, fault) for protocol in protocols for fault in faults]
print(f"{'protocol':<20} {'crash_at':<22} {'status':<10} commits  requests")
for row in report:
    print(f"{row['protocol']:<20} {row['crash_at']:<22} {row['status']:<10} {row['business_commits']:>7} {row['write_requests']:>9}")
assert trial('naive_replay', 'after_remote_commit')['business_commits'] == 2
assert all(row['business_commits'] == 1 for row in report if row['protocol'] != 'naive_replay')
assert trial('idempotent_replay', 'after_remote_commit')['write_requests'] == 2
assert trial('query_then_retry', 'after_remote_commit')['write_requests'] == 1


## 2 · 故意破坏协议

保持远端幂等实现不变，只在重试时换一个操作 ID。预测业务提交次数，再运行。接着模拟查询不可用：此时应保留 unknown，不能把无法查询当作“明确未提交”。

In [ ]:
changed_id = trial('idempotent_replay', 'after_remote_commit', reuse_id=False)
unavailable = trial('query_then_retry', 'after_remote_commit', query_available=False)
print('Changed retry identity:', changed_id)
print('Query unavailable:', unavailable)
assert changed_id['business_commits'] == 2
assert unavailable['status'] == 'unknown'
assert unavailable['business_commits'] == 1


## 3 · 同一 ID 的并发写入

20 个线程在屏障后同时调用。先预测资源数，再运行并观察同一回执、参数冲突、重新打开数据库后的重放和提交前异常回滚。

每个调用使用独立连接。`BEGIN IMMEDIATE` 获取写事务；操作记录、参数检查与资源写入一起提交。这个最小实现没有鉴权、多租户、幂等键过期或远端调用；它验证同库事务边界。


In [ ]:
import sqlite3
import tempfile
from concurrent.futures import ThreadPoolExecutor
from threading import Barrier

def initialize_database(path):
    with sqlite3.connect(path) as connection:
        connection.executescript("""
            CREATE TABLE actions (
                operation_id TEXT PRIMARY KEY NOT NULL,
                payload TEXT NOT NULL,
                receipt INTEGER
            );
            CREATE TABLE resources (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                operation_id TEXT UNIQUE NOT NULL,
                payload TEXT NOT NULL
            );
        """)

def transactional_write(path, operation_id, payload, crash_before_commit=False):
    canonical = json.dumps(payload, sort_keys=True, separators=(',', ':'))
    connection = sqlite3.connect(path, timeout=15)
    try:
        connection.execute('BEGIN IMMEDIATE')
        connection.execute('INSERT OR IGNORE INTO actions VALUES (?, ?, NULL)',
                           (operation_id, canonical))
        original, receipt = connection.execute(
            'SELECT payload, receipt FROM actions WHERE operation_id = ?',
            (operation_id,)).fetchone()
        if original != canonical:
            raise ValueError('Same operation ID, different payload')
        if receipt is None:
            cursor = connection.execute('INSERT INTO resources (operation_id, payload) VALUES (?, ?)',
                                        (operation_id, canonical))
            receipt = cursor.lastrowid
            connection.execute('UPDATE actions SET receipt = ? WHERE operation_id = ?',
                               (receipt, operation_id))
        if crash_before_commit:
            raise SimulatedCrash('Raised before the transaction commits')
        connection.commit()
        return receipt
    except Exception:
        connection.rollback()
        raise
    finally:
        connection.close()

def resource_count(path):
    with sqlite3.connect(path) as connection:
        return connection.execute('SELECT COUNT(*) FROM resources').fetchone()[0]

with tempfile.TemporaryDirectory() as directory:
    database = str(Path(directory) / 'operations.sqlite')
    initialize_database(database)
    workers = 20
    gate = Barrier(workers)
    def concurrent_call(_):
        gate.wait(timeout=15)
        return transactional_write(database, 'order-42', {'amount': 100})
    with ThreadPoolExecutor(max_workers=workers) as pool:
        receipts = list(pool.map(concurrent_call, range(workers)))
    commits = resource_count(database)
    assert commits == 1 and len(set(receipts)) == 1
    try:
        transactional_write(database, 'order-42', {'amount': 200})
        payload_conflict = False
    except ValueError:
        payload_conflict = True
    assert payload_conflict
    # Every call opens a new connection: replay does not rely on a Python cache.
    assert transactional_write(database, 'order-42', {'amount': 100}) == receipts[0]
    reopened = resource_count(database)
    rollback_database = str(Path(directory) / 'rollback.sqlite')
    initialize_database(rollback_database)
    try:
        transactional_write(rollback_database, 'order-new', {'amount': 100}, True)
    except SimulatedCrash:
        pass
    rollback_count = resource_count(rollback_database)
    assert reopened == 1 and rollback_count == 0
    assert transactional_write(rollback_database, 'order-new', {'amount': 100}) == 1
    assert resource_count(rollback_database) == 1
    sqlite_report = {'workers': workers, 'business_commits': commits,
                     'distinct_receipts': len(set(receipts)), 'payload_conflict': payload_conflict,
                     'reopen_commits': reopened, 'rollback_commits': rollback_count}
    print('SQLite transaction experiment:', sqlite_report)


## 4 · 把观察带回学习笔记

填写 `my_explanation` 后运行最后一格，从 Colab 文件侧栏下载 `recovery_report.json`，返回专题点击“导入实验结果”。已有笔记会保留，结果会追加在后面。

写清三点：哪种协议少发一次写请求；为什么同 ID 的 20 个并发调用只创建一条资源；一个你实际运行的反例和恢复策略。把 SQLite 提交前异常检查与真实进程被终止区分开。


In [ ]:
my_explanation = ""  # Replace with your own reasoning.
export = {'experiment': 'durable-agent', 'data_scope': 'synthetic in-memory simulation',
          'schema_version': 2, 'results': report, 'sqlite': sqlite_report, 'changed_id': changed_id, 'query_unavailable': unavailable,
          'my_explanation': my_explanation}
Path('recovery_report.json').write_text(json.dumps(export, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved recovery_report.json. Download it from the Colab Files panel if you want to keep the result.')
